In [35]:
!pip -q install git+https://github.com/huggingface/transformers
!pip -q install accelerate peft trl bitsandbytes datasets scikit-learn pillow

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [36]:
import os
import re
import random
import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2_5_VLForConditionalGeneration,
    Trainer,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [37]:
# -------------------------
# Global
# -------------------------
SEED = 42

# -------------------------
# Data split
# -------------------------
VALID_RATIO = 0.10
PSEUDO_MIN_VOTES = 4
USE_TIE_FREE_ONLY = True

# -------------------------
# Model
# -------------------------
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
OUTPUT_DIR = "/content/qwen25vl_pilot_v2"

# -------------------------
# Quantization
# -------------------------
LOAD_IN_4BIT = True
BNB_4BIT_COMPUTE_DTYPE = torch.bfloat16
BNB_4BIT_QUANT_TYPE = "nf4"
BNB_4BIT_USE_DOUBLE_QUANT = True

# -------------------------
# Image
# -------------------------
MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28

# -------------------------
# LoRA
# -------------------------
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# -------------------------
# Train
# -------------------------
EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

LOGGING_STEPS = 20
EVAL_STEPS = 100
SAVE_STEPS = 100

# -------------------------
# Pilot subset
# -------------------------
PILOT_MODE = True
PILOT_TRAIN_SIZE = 1500
PILOT_VALID_SIZE = 400

# -------------------------
# Generation
# -------------------------
MAX_NEW_TOKENS = 4

In [55]:
#본학습 v1 파라미터
PILOT_MODE = False
OUTPUT_DIR = "/content/qwen25vl_full_v1"

EPOCHS = 1
LEARNING_RATE = 8e-5

LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 1024 * 28 * 28

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM = 8

WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

LOGGING_STEPS = 50
EVAL_STEPS = 250
SAVE_STEPS = 250

In [38]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

In [56]:
from collections import Counter

train = pd.read_csv("/content/train.csv")
dev = pd.read_csv("/content/dev.csv")
test = pd.read_csv("/content/test.csv")

print("train:", train.shape)
print("dev:", dev.shape)
print("test:", test.shape)

train["answer"] = train["answer"].astype(str).str.strip().str.lower()

def get_qtype(q: str) -> str:
    q = str(q)

    if "몇 개" in q or "개수" in q:
        return "count"
    if "재질" in q or "소재" in q:
        return "material"
    if "색" in q or "색상" in q:
        return "color"
    if "어디" in q or "위치" in q:
        return "location"
    if "무엇" in q or "종류" in q or "어떤" in q:
        return "type"
    return "other"

train["qtype"] = train["question"].apply(get_qtype)
dev["qtype"] = dev["question"].apply(get_qtype)
test["qtype"] = test["question"].apply(get_qtype)

train["stratify_key"] = train["qtype"] + "_" + train["answer"]

train_df, valid_df = train_test_split(
    train,
    test_size=VALID_RATIO,
    random_state=SEED,
    stratify=train["stratify_key"]
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

answer_cols = ["answer1", "answer2", "answer3", "answer4", "answer5"]

def get_pseudo_info(row):
    vals = []
    for c in answer_cols:
        v = str(row[c]).strip().lower()
        if v in ["a", "b", "c", "d"]:
            vals.append(v)

    if len(vals) == 0:
        return pd.Series([None, 0, 0, True])

    cnt = Counter(vals)
    top_label, top_n = cnt.most_common(1)[0]
    is_tie = sum(1 for _, n in cnt.items() if n == top_n) > 1

    if USE_TIE_FREE_ONLY and is_tie:
        pseudo = None
    else:
        pseudo = top_label

    return pd.Series([pseudo, top_n, len(vals), is_tie])

dev[["pseudo_answer", "agree_n", "n_valid", "is_tie"]] = dev.apply(get_pseudo_info, axis=1)

dev_strong = dev[
    dev["pseudo_answer"].notna() &
    (dev["agree_n"] >= PSEUDO_MIN_VOTES)
].copy().reset_index(drop=True)

train_df.to_csv("/content/train_split.csv", index=False, encoding="utf-8-sig")
valid_df.to_csv("/content/valid_split.csv", index=False, encoding="utf-8-sig")
dev_strong.to_csv("/content/dev_pseudo_strong.csv", index=False, encoding="utf-8-sig")

print("\n[Train answer distribution]")
print(train["answer"].value_counts().sort_index())

print("\n[Train qtype distribution]")
print(train["qtype"].value_counts())

print("\n[Valid qtype distribution]")
print(valid_df["qtype"].value_counts())

print("\n[Strong pseudo-label distribution]")
print(dev_strong["pseudo_answer"].value_counts().sort_index())

train: (5073, 8)
dev: (4413, 12)
test: (5074, 7)

[Train answer distribution]
answer
a    1263
b    1265
c    1311
d    1234
Name: count, dtype: int64

[Train qtype distribution]
qtype
count       1745
material    1611
type        1203
color        368
location     116
other         30
Name: count, dtype: int64

[Valid qtype distribution]
qtype
count       175
material    162
type        120
color        36
location     12
other         3
Name: count, dtype: int64

[Strong pseudo-label distribution]
pseudo_answer
a    184
b    188
c    117
d    413
Name: count, dtype: int64


In [57]:
train_df = pd.read_csv("/content/train_split.csv")
valid_df = pd.read_csv("/content/valid_split.csv")

if PILOT_MODE:
    train_df = train_df.sample(
        n=min(PILOT_TRAIN_SIZE, len(train_df)),
        random_state=SEED
    ).reset_index(drop=True)

    valid_df = valid_df.sample(
        n=min(PILOT_VALID_SIZE, len(valid_df)),
        random_state=SEED
    ).reset_index(drop=True)

print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)
train_df.head()

train_df: (4565, 10)
valid_df: (508, 10)


,id,path,question,a,b,c,d,answer,qtype,stratify_key
0,train_4726.jpg,train/train_4726.jpg,사진에 보이는 재활용 가능한 플라스틱 병의 개수는 몇 개인가요?,1개,4개,2개,3개,c,count,count_c
1,train_4513.jpg,train/train_4513.jpg,사진에 보이는 재활용품 중 종이 재질인 것은 무엇인가요?,흰색 종이봉투,흰색 플라스틱 용기,나무 책상,검은색 금속 기둥,a,material,material_a
2,train_1570.jpg,train/train_1570.jpg,사진에 보이는 재활용품 중 투명한 플라스틱 용기의 개수는 몇 개인가요?,3개,4개,1개,2개,c,count,count_c
3,train_2757.jpg,train/train_2757.jpg,사진 속 아이스크림 컵의 재활용 분류는 무엇인가요?,유리병,종이류,캔류,플라스틱류,b,type,type_b
4,train_4585.jpg,train/train_4585.jpg,사진에 보이는 재활용품 중 종이 재질의 재활용품은 무엇인가요?,플라스틱 통,종이상자와 종이뭉치,페인트 통,금속 사다리,b,material,material_b


In [58]:
def resolve_image_path(path: str) -> str:
    path = str(path)
    if path.startswith("/content/"):
        return path
    return f"/content/{path}"

def build_prompt(question, a, b, c, d):
    return f"""이미지를 보고 객관식 문제의 정답만 고르세요.
반드시 a, b, c, d 중 하나의 문자만 출력하세요.
설명은 절대 쓰지 마세요.

질문: {question}
선택지:
a. {a}
b. {b}
c. {c}
d. {d}
"""

def extract_choice(text):
    text = str(text).strip().lower()
    m = re.search(r"\b([abcd])\b", text)
    if m:
        return m.group(1)
    if text in ["a", "b", "c", "d"]:
        return text
    return None

In [59]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_compute_dtype=BNB_4BIT_COMPUTE_DTYPE,
    bnb_4bit_quant_type=BNB_4BIT_QUANT_TYPE,
    bnb_4bit_use_double_quant=BNB_4BIT_USE_DOUBLE_QUANT,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

trainable params: 95,178,752 || all params: 8,387,345,408 || trainable%: 1.1348


In [60]:
class SimpleListDataset(Dataset):
    def __init__(self, df):
        self.rows = df.to_dict("records")

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        return {
            "row": row,
            "answer": str(row["answer"]).strip().lower()
        }

train_dataset = SimpleListDataset(train_df)
valid_dataset = SimpleListDataset(valid_df)

len(train_dataset), len(valid_dataset)

(4565, 508)

In [61]:
class QwenVLCollatorAnswerOnly:
    def __init__(self, processor):
        self.processor = processor
        self.pad_token_id = processor.tokenizer.pad_token_id

    def __call__(self, features):
        prompt_texts = []
        full_texts = []
        images = []

        for f in features:
            row = f["row"]
            answer = str(f["answer"]).strip().lower()

            image_path = resolve_image_path(row["path"])
            image = Image.open(image_path).convert("RGB")

            prompt = build_prompt(
                row["question"], row["a"], row["b"], row["c"], row["d"]
            )

            prompt_messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path},
                        {"type": "text", "text": prompt},
                    ],
                }
            ]

            full_messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path},
                        {"type": "text", "text": prompt},
                    ],
                },
                {
                    "role": "assistant",
                    "content": [
                        {"type": "text", "text": answer}
                    ],
                },
            ]

            prompt_text = self.processor.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )

            full_text = self.processor.apply_chat_template(
                full_messages,
                tokenize=False,
                add_generation_prompt=False
            )

            prompt_texts.append(prompt_text)
            full_texts.append(full_text)
            images.append(image)

        prompt_batch = self.processor(
            text=prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        full_batch = self.processor(
            text=full_texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        input_ids = full_batch["input_ids"]
        attention_mask = full_batch["attention_mask"]
        labels = input_ids.clone()

        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)

        for i, plen in enumerate(prompt_lengths):
            labels[i, :plen] = -100

        if self.pad_token_id is not None:
            labels[input_ids == self.pad_token_id] = -100

        batch = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

        for k in ["pixel_values", "image_grid_thw"]:
            if k in full_batch:
                batch[k] = full_batch[k]

        return batch

collator = QwenVLCollatorAnswerOnly(processor)

In [62]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=True,
    remove_unused_columns=False,
    report_to="none",
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    load_best_model_at_end=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collator,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [63]:
trainer.train()
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [49]:
os.listdir(OUTPUT_DIR)

['tokenizer_config.json',
 'processor_config.json',
 'chat_template.jinja',
 'tokenizer.json',
 'training_args.bin',
 'checkpoint-100',
 'README.md',
 'adapter_config.json',
 'adapter_model.safetensors',
 'checkpoint-188']

In [50]:
del model
torch.cuda.empty_cache()

In [51]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

processor = AutoProcessor.from_pretrained(
    OUTPUT_DIR,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS
)

model.eval()
print("adapter loaded")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

adapter loaded


In [53]:
valid_eval_df = pd.read_csv("/content/valid_split.csv").reset_index(drop=True)

results = []

for _, row in valid_eval_df.iterrows():
    image_path = resolve_image_path(row["path"])
    image = Image.open(image_path).convert("RGB")

    prompt = build_prompt(
        row["question"], row["a"], row["b"], row["c"], row["d"]
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    gen_ids = output_ids[:, inputs.input_ids.shape[1]:]

    out_text = processor.batch_decode(
        gen_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )[0]

    pred = extract_choice(out_text)

    results.append({
        "id": row["id"],
        "gt": row["answer"],
        "pred": pred,
        "correct": pred == row["answer"],
        "raw_output": out_text
    })

res_df = pd.DataFrame(results)
print(res_df.head(20))
print("sample accuracy:", res_df["correct"].mean())

                id gt pred  correct raw_output
0   train_2697.jpg  b    b     True          b
1   train_0222.jpg  c    c     True          c
2   train_0552.jpg  a    a     True          a
3   train_4435.jpg  b    b     True          b
4   train_2525.jpg  b    b     True          b
5   train_1040.jpg  d    d     True          d
6   train_4537.jpg  d    b    False          b
7   train_0075.jpg  a    a     True          a
8   train_0470.jpg  d    d     True          d
9   train_0042.jpg  b    b     True          b
10  train_4758.jpg  a    a     True          a
11  train_4892.jpg  a    a     True          a
12  train_1068.jpg  d    d     True          d
13  train_1462.jpg  d    d     True          d
14  train_3760.jpg  a    a     True          a
15  train_0143.jpg  b    c    False          c
16  train_2746.jpg  d    d     True          d
17  train_2662.jpg  c    c     True          c
18  train_0898.jpg  a    a     True          a
19  train_1790.jpg  b    b     True          b
sample accura